In [54]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [55]:
df=pd.read_csv('data.csv')

In [56]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4657 entries, 0 to 4656
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   index    4657 non-null   int64
 1   title    4657 non-null   str  
 2   genre    4657 non-null   str  
 3   summary  4657 non-null   str  
dtypes: int64(1), str(3)
memory usage: 9.6 MB


In [57]:
df.duplicated().value_counts()

False    4657
Name: count, dtype: int64

In [3]:
#df

In [4]:
#df['genre'].unique()

In [5]:
#df['title'].duplicated().value_counts()

In [61]:
#df=df.drop_duplicates(subset='title',keep='first')

In [6]:
#df.info()

In [7]:
#df['title'].duplicated().value_counts()

In [64]:
df['tags'] = df['genre'] + " " + df['summary']
df[['title', 'tags']].head()

,title,tags
0,Drowned Wednesday,fantasy Drowned Wednesday is the first Truste...
1,The Lost Hero,"fantasy As the book opens, Jason awakens on a..."
2,The Eyes of the Overworld,fantasy Cugel is easily persuaded by the merc...
3,Magic's Promise,fantasy The book opens with Herald-Mage Vanye...
4,Taran Wanderer,fantasy Taran and Gurgi have returned to Caer...


In [8]:
#df.info()

In [66]:
df = df[['index', 'title', 'tags']]
df.info()

<class 'pandas.DataFrame'>
Index: 4296 entries, 0 to 4656
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   index   4296 non-null   int64
 1   title   4296 non-null   str  
 2   tags    4296 non-null   str  
dtypes: int64(1), str(2)
memory usage: 9.3 MB


In [67]:
df

,index,title,tags
0,0,Drowned Wednesday,fantasy Drowned Wednesday is the first Truste...
1,1,The Lost Hero,"fantasy As the book opens, Jason awakens on a..."
2,2,The Eyes of the Overworld,fantasy Cugel is easily persuaded by the merc...
3,3,Magic's Promise,fantasy The book opens with Herald-Mage Vanye...
4,4,Taran Wanderer,fantasy Taran and Gurgi have returned to Caer...
...,...,...,...
4651,4651,Fantastic Beasts and Where to Find Them: The O...,fantasy J.K. Rowling's screenwriting debut is ...
4652,4652,Hounded,"fantasy Atticus O’Sullivan, last of the Druids..."
4653,4653,Charlie and the Chocolate Factory,fantasy Charlie Bucket's wonderful adventure b...
4654,4654,Red Rising,"fantasy ""I live for the dream that my children..."


In [68]:
from nltk.stem.porter import PorterStemmer
ps=PorterStemmer()

def stem_text(text):
    words = text.split()

    stemmed_words = []

    for word in words:
        stemmed_words.append(ps.stem(word))

    return " ".join(stemmed_words)

In [69]:
df['tags'] = df['tags'].str.lower()

In [70]:
df['tags']=df['tags'].apply(stem_text)

In [9]:
#df.head()

In [72]:
df.to_csv("clean_data_books.csv",index=False)

In [73]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf= TfidfVectorizer(max_features=5000, stop_words='english')

In [74]:
vectors = tfidf.fit_transform(df['tags']).toarray()

In [75]:
vectors.shape

(4296, 5000)

In [76]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)

In [77]:
similarity.shape

(4296, 4296)

In [78]:
#getting bookname by index
def get_name_by_index(i):
    if i < len(df) and i>0:
        return df.loc[i,'title']
    else:
        return''

m=get_name_by_index(12)
m

'Master Alvin'

In [82]:
# Function to get book index from book title
def get_book_index(book_name):

    found_index = -1

    for i in df.index:

        if df.loc[i, 'title'].lower() == book_name.lower():

            found_index = i
            break

    return found_index


# Function to get book title from index
def get_book_name(index):

    return df.loc[index, 'title']


# ---------------------- Recommendation ----------------------

name = input("Enter book name: ")

index = get_book_index(name)

if index != -1:

    similarity_indexes = list(enumerate(similarity[index]))

    similarity_indexes = sorted(
        similarity_indexes,
        key=lambda x: x[1],
        reverse=True
    )

    print("\nBecause you read:", df.loc[index, 'title'])
    print("\nYou may also like:\n")

    # Top 5 recommendations
    for i in range(1, 6):

        book_idx = similarity_indexes[i][0]

        print(i, ":", get_book_name(book_idx))

else:

    print("Book not found")

Enter book name:  percy jackson


Book not found


In [83]:
import pickle as pkl
pickle.dump(df, open("books.pkl", "wb"))
pkl.dump(similarity,open('similarity_books.pkl','wb'))